# Self-attention: от одного токена к контексту

[Подробная лекция: интуиция, математика и вопросы](../../notes/interview-prep/08-self-attention.md). Для остановки A прочитай разделы 1–6, для B — 7–9, для C — 10–12.

Работаем по трём остановкам: **A — Q/K/V**, **B — scores и softmax**, **C — смешивание Values**. После каждой остановки сохраняй notebook и присылай на проверку. Не нужно заполнять всё за один раз.

Результат занятия: самостоятельно реализовать одну голову self-attention и проверить её числа, формы и градиенты. Это часть Transformer: causal mask, выходная проекция, residual, LayerNorm и MLP появятся позже.

В этом уроке attention двунаправленный: каждая позиция видит все позиции своей последовательности. Для предсказания следующего токена такое внимание без causal mask использовать нельзя.

## A1. Восстанавливаем вход из прошлого урока

Batch — одна группа из B последовательностей. T — количество позиций в каждой; D — длина вектора позиции. Одинаковые параметры используются для всех элементов batch.

Token lookup выбирает строку по ID, position lookup — по номеру позиции. Их сумма содержит информацию о токене и позиции. Связи с окружающими токенами ещё предстоит вычислить.

Ячейка ниже — готовая демонстрация. Этот notebook не зависит от состояния предыдущих notebooks.

In [2]:
import math
import torch
from torch import nn

torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)

vocabulary = {"<pad>": 0, "кот": 1, "съел": 2, "сыр": 3}
token_ids = torch.tensor([[1, 2, 3], [3, 2, 1]], dtype=torch.long)
B, T = token_ids.shape
D = 4
max_context_length = 16

token_embedding = nn.Embedding(len(vocabulary), D)
position_embedding = nn.Embedding(max_context_length, D)
positions = torch.arange(T, device=token_ids.device)

X = token_embedding(token_ids) + position_embedding(positions)
print("IDs:", token_ids.shape)
print("X:", X.shape)
assert X.shape == (2, 3, 4)

IDs: torch.Size([2, 3])
X: torch.Size([2, 3, 4])


## A2. Зачем три представления

Для каждого входного вектора x создаём q, k и v тремя разными линейными слоями:

- Query участвует в выборе источников информации.
- Key сравнивается с Query и определяет вес источника.
- Value содержит информацию, которая будет смешиваться с этими весами.

Метафора «что я ищу / как меня найти / что передаю» помогает запомнить роли, но это не заранее заданные языковые признаки. Здесь веса случайные, смысловые связи ещё не выучены.

Все проекции пока имеют размер D → D. Для параметров PyTorch верна запись **Q = X @ query_projection.weight.T**; аналогично для K и V. Bias отключён. Каждая строка матрицы весов принадлежит одному выходному нейрону.

На этом шаге каждая позиция обрабатывается независимо. Обмен информацией между позициями произойдёт позже.

In [3]:
# Создаём слои здесь; при повторе этой ячейки они инициализируются заново.
query_projection = nn.Linear(D, D, bias=False)
key_projection = nn.Linear(D, D, bias=False)
value_projection = nn.Linear(D, D, bias=False)

In [4]:
# TODO: вызови соответствующие слои на X.
Q = X @ query_projection.weight.T
K = X @ key_projection.weight.T
V = X @ value_projection.weight.T

assert isinstance(Q, torch.Tensor), "Вычисли Q через query_projection(X)"
assert isinstance(K, torch.Tensor), "Вычисли K через key_projection(X)"
assert isinstance(V, torch.Tensor), "Вычисли V через value_projection(X)"
assert Q.shape == K.shape == V.shape == (B, T, D)

print("Q первого токена:", Q[0, 0])
print("K первого токена:", K[0, 0])
print("V первого токена:", V[0, 0])

# Проверка того, что nn.Linear выполняет знакомое матричное умножение.
torch.testing.assert_close(Q, query_projection(X))
torch.testing.assert_close(K, key_projection(X))
torch.testing.assert_close(V, value_projection(X))

Q первого токена: tensor([-0.032,  1.168, -0.707,  0.725], grad_fn=<SelectBackward0>)
K первого токена: tensor([ 1.957, -0.380,  1.053, -0.952], grad_fn=<SelectBackward0>)
V первого токена: tensor([1.379, 1.260, 0.676, 0.162], grad_fn=<SelectBackward0>)


### Остановка A — объясни своими словами

1. Что означают B=2, T=3, D=4?
2. Почему Q, K и V одинаковы по форме, но могут отличаться по значениям?
3. Сколько параметров суммарно у трёх проекций без bias?
4. Начали ли токены обмениваться информацией при вычислении Q/K/V?

**Мои ответы:**

1. 2 последовательности, по 3 токена в каждой, размерерность токена =4, т.е. токен кодируется вектором размерности 4
2. Потому что отвечают за разные вещи. Каждая кодирует свой сигнал
3. 3*D*D
4. нет

Сохрани notebook и остановись здесь для первого разбора.

## B1. Одно сравнение вручную

Скалярное произведение — сумма произведений соответствующих координат. Query позиции i сравнивается с Key позиции j:

$$s_{ij}=q_i\cdot k_j=\sum_r q_{i,r}k_{j,r}$$

Например, q=[1,2], k=[3,4] дают score 1×3 + 2×4 = 11. Score — произвольное действительное число, ещё не вероятность. Это не обязательно cosine similarity: длины векторов тоже влияют на результат.

Для T токенов нужно T×T сравнений. Строка i отвечает за Query получателя, столбец j — за Key источника.

In [7]:
# TODO: вычисли raw_scores для всех пар позиций.
# Подсказки: матричное умножение @, K.transpose(-2, -1).
# (B, T, D) @ (B, D, T) -> (B, T, T)
raw_scores = Q @ K.transpose(-2, -1)

assert isinstance(raw_scores, torch.Tensor), "Вычисли матрицу сравнений"
assert raw_scores.shape == (B, T, T)

# Один элемент проверяем независимым поэлементным вычислением.
manual_score = (Q[0, 1] * K[0, 2]).sum()
torch.testing.assert_close(raw_scores[0, 1, 2], manual_score)
print("Scores первой последовательности:")
print(raw_scores[0])

Scores первой последовательности:
tensor([[-1.941,  1.661, -0.651],
        [-0.055, -0.796,  0.350],
        [-1.630,  1.034,  1.641]], grad_fn=<SelectBackward0>)


## B2. Scaling и softmax

Softmax превращает scores одной Query в положительные веса с суммой 1:

$$a_{ij}=\frac{\exp(s_{ij}/\sqrt{d_k})}{\sum_m\exp(s_{im}/\sqrt{d_k})}$$

Здесь d_k=D. При независимых координатах q и k с нулевым средним и единичной дисперсией дисперсия их скалярного произведения растёт как d_k. Деление на sqrt(d_k) удерживает масштаб scores и помогает избежать слишком резкого softmax. Это мотивировка при определённых предположениях, а не гарантия для любых векторов.

Суммируем по всем Keys для одной Query, поэтому softmax идёт по последнему измерению. Используем готовый torch.softmax, который вычисляет результат численно устойчиво.

In [10]:
# TODO: масштабируй scores и получи веса через torch.softmax(..., dim=-1).
scaled_scores = raw_scores / math.sqrt(Q.shape[-1])
attention_weights = torch.softmax(scaled_scores, dim=-1)

assert isinstance(attention_weights, torch.Tensor), "Вычисли softmax"
assert attention_weights.shape == (B, T, T)
assert torch.all(attention_weights >= 0)
torch.testing.assert_close(
    attention_weights.sum(dim=-1),
    torch.ones(B, T, device=X.device),
)
print(attention_weights[0])

tensor([[0.112, 0.676, 0.213],
        [0.343, 0.237, 0.420],
        [0.101, 0.382, 0.517]], grad_fn=<SelectBackward0>)


### Остановка B

1. Что означает raw_scores[0, 1, 2]?
2. Почему softmax нужен по последнему измерению?
3. Одинаковы ли score[i,j] и score[j,i] обязательно? Вспомни, что Q и K получены разными проекциями.
4. Если все scores строки одинаковы, какими будут её веса?

**Мои ответы:**

1. raw_scores[0, 1, 2] — скалярное произведение Query второго токена с Key третьего токена в первой последовательности. Индексы означают [номер последовательности, позиция Query, позиция Key]. Последний индекс здесь НЕ координата эмбеддинга: форма raw_scores — (B,T,T), а не (B,T,D). Это ещё не вес внимания.

Разбор на нашей последовательности [кот, съел, сыр]: raw_scores[0,1,2] = (Q[0,1] * K[0,2]).sum(). Мы умножаем соответствующие координаты двух векторов и складываем их, получая одно число. Например, для выдуманных q=[1,2] и k=[3,4] результат равен 1*3 + 2*4 = 11.

Raw означает исходные, ещё не обработанные scores: сначала Q @ K.transpose(-2,-1), затем деление на sqrt(d_k), затем softmax. Raw scores могут быть отрицательными, не являются вероятностями и не обязаны суммироваться в 1.

2. Последняя ось — позиции Keys, то есть источники информации для одной Query. Softmax по dim=-1 превращает одну строку сравнений в веса источников с суммой 1. Например, attention_weights[0,1] — все веса, с которыми второй токен первой последовательности смешивает Values.

3. Нет. score[i,j] = q_i · k_j, а score[j,i] = q_j · k_i. Q и K получены разными проекциями, поэтому это разные пары векторов. Коммутативность скалярного произведения даёт q_i · k_j = k_j · q_i, но не q_j · k_i. Например, q_i=[1,0], k_j=[2,0] дают 2; q_j=[0,1], k_i=[0,3] дают 3.

4. Без маски одинаковые scores дадут одинаковые веса: каждый равен 1/T. При T=3 получится [1/3,1/3,1/3], даже если исходная строка была [-5,-5,-5]. После экспоненты все значения равны, и каждое делится на сумму трёх одинаковых значений. С маской равномерность возможна по разрешённым позициям, если их scores одинаковы.


## C1. Сначала смешаем два Value вручную

Пусть одна Query уже получила веса [0.25, 0.75], а Values равны [2,0] и [0,4]. Тогда:

$$c=0.25[2,0]+0.75[0,4]=[0.5,3]$$

Одна позиция получила один новый вектор, собрав информацию от двух источников. Мы суммируем векторы, поэтому их длина не увеличивается.

In [11]:
weights_example = torch.tensor([0.25, 0.75])
values_example = torch.tensor([[2.0, 0.0], [0.0, 4.0]])

# TODO: получи вектор [0.5, 3.0] матричным умножением.
mixed_example = weights_example @values_example
assert isinstance(mixed_example, torch.Tensor)
torch.testing.assert_close(mixed_example, torch.tensor([0.5, 3.0]))

## C2. Все позиции одновременно

$$C=AV$$

$$c_{b,i}=\sum_j a_{b,i,j}v_{b,j}$$

Формы: (B,T,T) @ (B,T,D) → (B,T,D). Каждый выходной вектор связан с одной Query, но содержит смесь Values нескольких позиций. Разные последовательности batch не смешиваются.

В общем случае размер Value может отличаться от размера Query/Key. Здесь мы намеренно выбрали все размеры равными D.

In [12]:
# TODO: смешай Values с полученными весами.
context =  attention_weights @ V

assert isinstance(context, torch.Tensor)
assert context.shape == (B, T, D)

# Проверим одну позицию через явную взвешенную сумму.
manual_context = (attention_weights[0, 1, :, None] * V[0]).sum(dim=0)
torch.testing.assert_close(context[0, 1], manual_context)
print("Context:", context.shape)
print(context[0])

Context: torch.Size([2, 3, 4])
tensor([[-1.103, -1.051, -0.521, -0.546],
        [-0.159, -0.288, -0.268, -0.558],
        [-0.792, -0.894, -0.603, -0.781]], grad_fn=<SelectBackward0>)


## C3. Видит ли градиент проекции?

Ниже техническая проверка связности графа, а не обучение языковой модели: искусственный loss нужен только для демонстрации backward. После backward параметры ещё не обновлены.

Повторный запуск этой ячейки требует заново выполнить forward-ячейки с X, Q/K/V, scores и context: первый backward освобождает сохранённые промежуточные данные.

In [13]:
modules = [token_embedding, position_embedding,
           query_projection, key_projection, value_projection]
for module in modules:
    module.zero_grad(set_to_none=True)

probe_loss = context.square().mean()
probe_loss.backward()

for name, module in zip(["token", "position", "Q", "K", "V"], modules):
    grad = module.weight.grad
    assert grad is not None and torch.isfinite(grad).all()
    print(name, "grad shape:", tuple(grad.shape), "norm:", grad.norm().item())

token grad shape: (4, 4) norm: 0.4520016610622406
position grad shape: (16, 4) norm: 0.4516163468360901
Q grad shape: (4, 4) norm: 0.5041609406471252
K grad shape: (4, 4) norm: 0.4285697340965271
V grad shape: (4, 4) norm: 0.8409233689308167


### Остановка C — интервью и выводы

1. Что делает self-attention, чего не делает обычный nn.Linear по последней оси?
2. Чем отличаются attention weights и обучаемые веса проекций?
3. Почему context имеет форму (B,T,D), а не (B,T,T)?
4. Почему случайные attention weights пока нельзя интерпретировать как осмысленные языковые связи?
5. Может ли текущая реализация подсматривать будущие токены?
6. Какие тензоры обновляет optimizer, а какие заново вычисляются при следующем forward?

**Мои ответы:**

1. Обычный nn.Linear по последней оси независимо преобразует вектор каждой позиции одной общей матрицей. Self-attention смешивает Values разных позиций внутри одной последовательности; веса смешивания вычисляются из её содержимого через Q и K. Поэтому результат позиции может зависеть от других токенов.

2. Веса проекций — обучаемые параметры слоёв, например query_projection.weight. Оптимизатор обновляет их, и они сохраняются в модели. Attention weights — промежуточный результат softmax(QK^T/sqrt(d_k)) для конкретного входа, а не отдельные параметры оптимизатора. Они вычисляются заново при каждом forward и могут различаться для разных текстов даже при фиксированных параметрах модели.

3. В attention_weights формы (B,T,T) для каждой Query хранится по одному весу на каждый источник. Затем эти веса используются для суммы Value-векторов: (B,T,T) @ (B,T,D) -> (B,T,D). Мы складываем векторы, а не склеиваем их: на каждую позицию остаётся один вектор длины D. В общем случае длина результата — d_v, если Values имеют размерность d_v.

4. Эмбеддинги и проекции пока случайно инициализированы и не обучались полезной задаче. Поэтому высокий вес сам по себе не доказывает найденную грамматическую или смысловую связь. Даже у обученной модели веса внимания не являются полным объяснением предсказания: важны также Values и остальные вычисления.

5. Да. Causal mask отсутствует, поэтому Query каждой позиции сравнивается в том числе с Keys последующих позиций и может смешивать их Values. Для предсказания следующего токена при обучении такие будущие позиции нужно маскировать до softmax.

6. Оптимизатор обновляет переданные ему обучаемые параметры. В нашей модели это могут быть таблицы token/position embeddings и матрицы Q/K/V-проекций. X, Q, K, V, scores, attention_weights и context вычисляются заново при следующем forward. В текущем notebook optimizer.step() нет: probe_loss.backward() только вычисляет градиенты; искусственный loss проверяет связность графа, а не учит предсказывать текст.

После выполнения перезапусти kernel и выполни все ячейки по порядку. Проверки должны пройти. Outputs сохраняй для разбора.

Следующий этап: causal mask, затем соберём attention в nn.Module и встроим его в маленькую языковую модель.